---
# **Project 1: Frog Tail - Applied Data Science (Fall 2025)**
---



**<font color="brown">MARÍA DOLORES NAVARRO MATURANA (LOLA)  -  mn3312</font>**

Loads cleaned frog-tail scRNA-seq, normalizes/log1p, selects HVGs, PCA→neighbors→UMAP, clusters (Leiden/Louvain), evaluates (ARI/NMI/Silhouette), scores ROC markers (TP63/LEF1/Krt), performs DE, light annotation, QC panels, and packages outputs.

In [ ]:
%pip -q install --upgrade \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "pandas==2.2.2" \
  "anndata==0.12.2" \
  "scanpy==1.10.2" \
  "scikit-learn==1.4.2" \
  "python-igraph" "leidenalg" \
  "bbknn==1.6.0" "harmonypy==0.0.9" \
  "magic-impute==3.0.0"

## **1) Loading & preprocessing**

I start from the cleaned frog-tail dataset and implement my pipeline step-by-step.

### Load Data

In [ ]:
import scanpy as sc
import anndata as ad
import sys
import pandas as pd
import numpy as np
import scipy.sparse as sp
import random
import matplotlib.pyplot as plt
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import trustworthiness
import igraph as ig

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.stats import hypergeom
from pathlib import Path

import bbknn
import harmonypy as hm

random.seed(0)
np.random.seed(0)

sc.settings.figdir = "figures"

print("scanpy", sc.__version__, "| anndata", ad.__version__, "| python", sys.version.split()[0])


adata = ad.read_h5ad('/content/cleaned_processed_frogtail.h5ad')  #I have upload it to colab

In [ ]:
adata.obs

In [ ]:
adata

### Preprocessing

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalize to 1e4 UMIs per cell, then log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Save a log-normalized snapshot for DE/marker tools
adata.raw = adata.copy()

In [ ]:
# Light filters and HVG selection
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)

sc.pp.highly_variable_genes(adata, n_top_genes=2300)

In [ ]:
sc.pl.highly_variable_genes(adata)

## **2) Dimensionality reduction & graph**

In [ ]:
# scale -> PCA -> neighbors -> UMAP

#scale
sc.pp.scale(adata, max_value=10)

# PCA - PCA on HVGs; arpack keeps results stable across runs
sc.tl.pca(adata, svd_solver="arpack", mask_var="highly_variable", random_state=0)
sc.pl.pca_variance_ratio(adata, log=True)

# neighbors - build kNN graph (15 neighbors, 40 PCs worked best here)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40, random_state=0)

# UMAP for visualization
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color=["sample","batch","DevelopmentalStage","DaysPostAmputation"])


## **3) Clustering (Leiden + Louvain)**

In [ ]:
# Leiden clustering: try a few resolutions and pick the best by silhouette
resolutions = [0.4, 0.8, 1.0, 1.2]
results = []

for r in resolutions:
    key = f"leiden_{r:.1f}"
    sc.tl.leiden(adata, resolution=r, key_added=key, random_state=0)
    # Metrics
    # silhouette on PCA space (more stable than UMAP for metric calculations)
    sil = silhouette_score(adata.obsm["X_pca"], adata.obs[key].astype("category").cat.codes)
    results.append({"resolution": r, "silhouette_pca": sil})

results_df = pd.DataFrame(results)
results_df

# Pick the Leiden resolution with the best silhouette
best_row = max(results, key=lambda d: d["silhouette_pca"])
best_res = best_row["resolution"]
best_key = f"leiden_{best_res:.1f}"
print(f"Best Leiden resolution by silhouette: {best_res}  (key: {best_key})")


# Louvain (igraph flavor) for a second view of clusters
sc.tl.louvain(adata, key_added="louvain_igraph", flavor="igraph", random_state=0)

# UMAP
sc.pl.umap(adata, color=[best_key, "louvain_igraph"], wspace=0.4)


# Save AnnData with clustering labels
adata.write_h5ad("/content/frogtail_processed_with_clusters.h5ad", compression="gzip")

# Save UMAP figures
sc.pl.umap(adata, color=[best_key], save=f"_{best_key}.png", show=False)
sc.pl.umap(adata, color=["louvain_igraph"], save="_louvain_igraph.png", show=False)


## **4) Clustering agreement metrics**

In [ ]:
# Record the chosen Leiden partition
adata.obs["leiden_best"] = adata.obs[best_key].astype("category")

# Clustering-agreement metrics
ari = adjusted_rand_score(adata.obs["leiden_best"], adata.obs["louvain_igraph"])
nmi = normalized_mutual_info_score(adata.obs["leiden_best"], adata.obs["louvain_igraph"])
sil = silhouette_score(adata.obsm["X_pca"], adata.obs["leiden_best"].cat.codes)

metrics = pd.DataFrame(
    {"metric": ["ARI (Leiden vs Louvain)", "NMI (Leiden vs Louvain)", "Silhouette (PCA, Leiden)"],
     "value": [ari, nmi, sil]}
)
metrics


## **5) ROC marker enrichment (TP63/LEF1/Krt)**

In [ ]:
# Make sure the cluster labels are categorical
adata.obs[best_key] = adata.obs[best_key].astype("category")

# Candidate marker names (include Xenopus .L/.S alleles + common keratins)
candidates = [
    "Tp63", "Tp63.L", "Tp63.S",
    "LEF1", "Lef1", "Lef1.L", "Lef1.S",
    "Krt", "Krt.L", "krt8", "krt18", "KRT8", "KRT18"
]

# Case-insensitive match against adata.var_names
vn_lower = pd.Index([v.lower() for v in adata.var_names])
present = []
for g in candidates:
    if g.lower() in vn_lower:
        present.append(adata.var_names[vn_lower.get_loc(g.lower())])

print(f"ROC markers found in matrix ({len(present)}):", present)

# Compute a composite ROC score and visualize
sc.tl.score_genes(adata, gene_list=present, score_name="ROC_score")

sc.pl.umap(adata, color=["ROC_score"], cmap="viridis")
sc.pl.violin(adata, keys="ROC_score", groupby=best_key, rotation=90, stripplot=False)

# Select the Leiden cluster with the highest average ROC score
roc_means = (
    adata.obs.groupby(best_key, observed=True)["ROC_score"]
    .mean()
    .sort_values(ascending=False)
)
roc_cluster = roc_means.index[0]
print("Selected ROC cluster (highest mean ROC_score):", roc_cluster)

adata.uns["roc_marker_candidates"] = present
adata.uns["roc_selected_cluster"]   = str(roc_cluster)
adata.uns["roc_best_key"]           = best_key


# Flag cells in the ROC cluster and plot
adata.obs["roc_cluster_flag"] = (adata.obs[best_key] == roc_cluster).astype(int)
sc.pl.umap(adata, color=[best_key, "roc_cluster_flag"], wspace=0.4)

# Save artifacts (H5AD + figures)
adata.write_h5ad("/content/frogtail_with_ROCscore.h5ad", compression="gzip")
sc.pl.umap(adata, color=["ROC_score"], save="_ROC_score.png", show=False)
sc.pl.violin(
    adata, keys="ROC_score", groupby=best_key, rotation=90,
    stripplot=False, save="_ROC_by_cluster.png", show=False
)

## **6) Differential expression: ROC cluster vs rest (Wilcoxon)**


In [ ]:
# Test the ROC-selected Leiden cluster against the rest
grp = str(roc_cluster)
gby = best_key           # for example leiden_0.4

# Wilcoxon rank-sum on the log1p-normalized counts
sc.tl.rank_genes_groups(
    adata,
    groupby=gby,
    groups=[grp],
    reference="rest",
    method="wilcoxon",
    use_raw=True,
    n_genes=200,      # compute more than we plot
    pts=True,         # also compute % cells expressing
)

# Quick overview plots
sc.pl.rank_genes_groups(adata, groups=[grp], n_genes=20, sharey=False)

# Tidy results into a DataFrame and save
de_df = sc.get.rank_genes_groups_df(adata, group=grp)
de_df.to_csv(f"/content/DE_{gby}_{grp}_vs_rest.csv", index=False)
de_df.head(10)


## **7) Cluster-level markers & quick labels**

In [ ]:
# Find marker genes per Leiden cluster (Wilcoxon on log1p-normalized counts)
sc.tl.rank_genes_groups(
    adata,
    groupby=gby,
    method="wilcoxon",
    use_raw=True,     # uses the normalized snapshot I stored earlier
    n_genes=50,
    pts=True
)

# Quick look at top markers
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

# Collect all marker results into a single DataFrame
markers_df = sc.get.rank_genes_groups_df(adata, None)  # all groups
markers_df.head(10)

# Save full marker table
markers_df.to_csv(f"/content/markers_{gby}.csv", index=False)

# Keep cluster IDs and tag the ROC cluster
label_map = {c: f"Cluster {c}" for c in adata.obs[gby].cat.categories}
label_map[str(roc_cluster)] = f"Cluster {roc_cluster} (ROC-candidate)"

adata.obs["annotation_major"] = adata.obs[gby].map(label_map).astype("category")

# UMAP with these simple annotations
sc.pl.umap(adata, color=["annotation_major"], wspace=0.4)


## **8) QC overlays + broad lineage panels**

In [ ]:
# QC: mito / ribosomal percentages computed on raw counts
X_counts = adata.layers["counts"] if "counts" in adata.layers else adata.X

def row_sum(arr):
    return arr.sum(axis=1).A1 if sp.issparse(arr) else arr.sum(axis=1)

vnames = pd.Index(adata.var_names)
vlow   = pd.Index([v.lower() for v in vnames])

mt_mask  = vlow.str.startswith(("mt-", "mt_", "mt.", "mt")) | vnames.str.upper().str.startswith("MT-")
rpl_mask = vnames.str.upper().str.startswith("RPL")
rps_mask = vnames.str.upper().str.startswith("RPS")

den = row_sum(X_counts).astype(float)
num_mt  = row_sum(X_counts[:, mt_mask]).astype(float)
num_rpl = row_sum(X_counts[:, rpl_mask]).astype(float)
num_rps = row_sum(X_counts[:, rps_mask]).astype(float)

with np.errstate(divide="ignore", invalid="ignore"):
    adata.obs["pct_counts_mt"]  = np.divide(num_mt,  den, out=np.zeros_like(den), where=den > 0) * 100.0
    adata.obs["pct_counts_rpl"] = np.divide(num_rpl, den, out=np.zeros_like(den), where=den > 0) * 100.0
    adata.obs["pct_counts_rps"] = np.divide(num_rps, den, out=np.zeros_like(den), where=den > 0) * 100.0

# Visual checks on UMAP
sc.pl.umap(adata, color=["pct_counts_mt","pct_counts_rpl","pct_counts_rps"], wspace=0.4)

# Quick lineage panels (case-insensitive, allow .L/.S alleles)
def present_panel(marker_list):
    """Return gene names from adata.var_names matching any markers (case-insensitive, .L/.S allowed)."""
    out = []
    for g in marker_list:
        base = g.lower()
        for cand in (base, f"{base}.l", f"{base}.s"):
            if cand in vlow:
                out.append(vnames[vlow.get_loc(cand)])
                break
    return sorted(set(out))

marker_sets = {
    # Skin / epidermis
    "Epidermal_Keratin": ["krt8","krt18","krt"],
    "Ionocyte":          ["atp6v1a","foxi1","cftr"],
    "Goblet":            ["muc1","muc2","spdef"],

    # Immune / RBC
    "Myeloid":           ["lyz","mpo","csf1r"],
    "Erythroid":         ["hba","hbb","sptb"],

    # Neural
    "Neural":            ["sox2","tubb3","elavl3","snap25","rbfox3"],

    # NEW: components that make up “Somite and others”
    # myotome / skeletal muscle
    "Somite_Myotome":    ["myod1","myf5","myog","acta1","tnnc2","myh1"],
    # mesenchyme / fibroblast-like
    "Somite_Mesenchyme": ["pdgfra","col1a1","col1a2","tagln"],
    # cartilage / sclerotome-ish
    "Somite_Cartilage":  ["sox9","col2a1","acana"],
    # endothelial / vascular
    "Somite_Endothelial":["kdr","kdrl","vegfr2","cdh5","pecam1"],
    # notochord (broad)
    "Somite_Notochord":  ["tbxt","shh","col2a1","lamc1"],
    # smooth muscle
    "Somite_Smooth":     ["tagln","myh11","acta2"]
}


gby = best_key  # for example leiden_0.4
score_cols = []
for label, raw_genes in marker_sets.items():
    genes = present_panel(raw_genes)
    if len(genes) == 0:
        continue
    col = f"score_{label}"
    sc.tl.score_genes(adata, gene_list=genes, score_name=col)
    score_cols.append(col)
    adata.uns[f"markers_{label}"] = genes  # keep which genes were actually used

# UMAPs and violins for any panels that hit
if score_cols:
    sc.pl.umap(adata, color=score_cols, wspace=0.4)
    sc.pl.violin(adata, keys=score_cols, groupby=gby, rotation=90, stripplot=False)
else:
    print("No marker panels matched; gene symbols may differ (OK).")

# Replace any '/' in adata.uns keys with '_'
bad_keys = [k for k in list(adata.uns.keys()) if "/" in k]
for k in bad_keys:
    adata.uns[k.replace("/", "_")] = adata.uns.pop(k)

adata.write_h5ad("/content/frogtail_part8_qc_and_panels.h5ad", compression="gzip")
for c in ["pct_counts_mt","pct_counts_rpl","pct_counts_rps"] + score_cols:
    sc.pl.umap(adata, color=[c], save=f"_{c}.png", show=False)


In [ ]:
# Part 8b: Broad-tissue label (Fig.1B-like, single UMAP)

def _has(col):
    return col in adata.obs.columns

# z-score each available panel score so panels with more genes don't dominate
panel_cols = [c for c in adata.obs.columns if c.startswith("score_")]
for c in panel_cols:
    col = adata.obs[c].astype(float)
    std = float(col.std(ddof=0)) or 1.0
    adata.obs[c + "_z"] = (col - float(col.mean())) / std

# group scores (use *_z versions when present)
def _z(c):
    return (c + "_z") if _has(c + "_z") else c

group_scores = {
    "Skin":              [ _z(c) for c in ["score_Epidermal_Keratin","score_Goblet","score_Ionocyte"] if _has(_z(c)) ],
    "Neural":            [ _z(c) for c in ["score_Neural"] if _has(_z(c)) ],
    "Immune":            [ _z(c) for c in ["score_Myeloid"] if _has(_z(c)) ],
    "Red blood cells":   [ _z(c) for c in ["score_Erythroid"] if _has(_z(c)) ],
    # sum of any somite-like panels you computed → approximates “Somite and others”
    "Somite and others": [ _z(c) for c in [
        "score_Somite_Myotome","score_Somite_Mesenchyme","score_Somite_Cartilage",
        "score_Somite_Endothelial","score_Somite_Notochord","score_Somite_Smooth"
    ] if _has(_z(c)) ],
}

import pandas as pd
S = pd.DataFrame(index=adata.obs_names)
for g, cols in group_scores.items():
    if len(cols) == 0:
        S[g] = 0.0
    elif len(cols) == 1:
        S[g] = adata.obs[cols[0]]
    else:
        S[g] = adata.obs[cols].sum(axis=1)

# assign label; if no group is positive, fall back to "Somite and others"
maxval = S.max(axis=1).fillna(0)
labels = S.idxmax(axis=1)
labels[maxval <= 0] = "Somite and others"

adata.obs["broad_tissue"] = pd.Categorical(labels)

# Save a Fig.1B-like UMAP (single panel)
sc.pl.umap(
    adata, color=["broad_tissue"], legend_loc="on data", frameon=False,
    save="_Figure1B_broad_tissue.png", show=False
)


In [ ]:
# Part 8c: Skin-only panel

skin_cols = [c for c in ["score_Epidermal_Keratin","score_Goblet","score_Ionocyte"] if c in adata.obs]
if len(skin_cols) > 0:
    # subset to skin-like cells
    skin_mask = (adata.obs[skin_cols].sum(axis=1) > 0).values
    ad_skin = adata[skin_mask].copy()

    # re-embed with slightly tighter parameters (purely cosmetic)
    sc.tl.pca(ad_skin, svd_solver="arpack", mask_var="highly_variable", random_state=0)
    sc.pp.neighbors(ad_skin, n_neighbors=12, n_pcs=min(40, ad_skin.obsm["X_pca"].shape[1]), random_state=0)
    sc.tl.umap(ad_skin, min_dist=0.3, spread=1.2, random_state=0)

    # quick(exportable) side-by-side like the paper
    sc.pl.umap(
        ad_skin,
        color=[best_key, "ROC_score"] if "ROC_score" in ad_skin.obs else [best_key],
        wspace=0.4, frameon=False, save="_Figure1B_skin_only.png", show=False
    )

    # ---- polish: label the ROC hotspot on the left panel ----
    if "ROC_score" in ad_skin.obs:
        thr = np.quantile(ad_skin.obs["ROC_score"], 0.98)
        roc_mask_skin = (ad_skin.obs["ROC_score"] >= thr).values
    else:
        roc_mask_skin = (ad_skin.obs[best_key].astype(str).values == str(roc_cluster))

    xy = ad_skin.obsm["X_umap"]
    x_lab, y_lab = xy[roc_mask_skin, 0].mean(), xy[roc_mask_skin, 1].mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    sc.pl.umap(ad_skin, color=[best_key], ax=axes[0], show=False, legend_loc="right margin", frameon=False)
    axes[0].set_title(best_key)
    axes[0].text(x_lab, y_lab, "ROC", fontsize=12, weight="bold")

    sc.pl.umap(ad_skin, color=["ROC_score"] if "ROC_score" in ad_skin.obs else [best_key],
               cmap="viridis", ax=axes[1], show=False, frameon=False)
    axes[1].set_title("ROC_score")

    outp = "figures/umap_Figure1B_skin_only_polished.png"
    fig.savefig(outp, dpi=200, bbox_inches="tight")
    print(f"[8c] Saved -> figures/umap_Figure1B_skin_only.png and {outp}")
else:
    print("[8c] No skin panel scores found; skipping skin-only panel.")


## **9) Second marker method: logistic regression (and compare to Wilcoxon)**


In [ ]:
gby = best_key
roc_str = str(roc_cluster)

# Logistic regression markers per cluster
sc.tl.rank_genes_groups(
    adata,
    groupby=gby,
    method="logreg",
    use_raw=True,
    n_genes=200,
    pts=True,
    max_iter=2000,  # <-- to avoid the warning
)
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

logreg_df = sc.get.rank_genes_groups_df(adata, None)  # all groups
logreg_df.to_csv(f"/content/markers_logreg_{gby}.csv", index=False)

# Build a ROC gene list by wilcoxon and logreg and compare overlap

top_n = 50
wil_co = (
    markers_df.query("group == @roc_str")
              .sort_values("scores", ascending=False)
              .head(top_n)["names"].str.lower()
)
lr_co = (
    logreg_df.query("group == @roc_str")
             .sort_values("scores", ascending=False)
             .head(top_n)["names"].str.lower()
)

wil_set = set(wil_co)
lr_set  = set(lr_co)

jaccard = len(wil_set & lr_set) / max(1, len(wil_set | lr_set))

overlap_df = pd.DataFrame({
    "metric": ["overlap_top50", "jaccard_top50"],
    "value":  [len(wil_set & lr_set), jaccard]
})
overlap_df.to_csv(f"/content/overlap_wilcoxon_vs_logreg_{gby}_ROC{roc_str}.csv", index=False)
overlap_df


## **10) Extra clustering metrics (CH / DB / Modularity + UMAP trustworthiness)**

In [ ]:
labels = adata.obs[best_key].cat.codes.values
X_pca  = adata.obsm["X_pca"]
X_umap = adata.obsm["X_umap"]

# Classic cluster-compactness/separation metrics (higher CH is better, lower DB is better)
db = davies_bouldin_score(X_pca, labels)
ch = calinski_harabasz_score(X_pca, labels)

# UMAP trustworthiness (higher is better): how well local neighborhoods are preserved from PCA space
tw = trustworthiness(X_pca, X_umap, n_neighbors=15)

# Graph modularity (higher is better): build an undirected igraph from Scanpy's kNN connectivities
C = adata.obsp["connectivities"].tocsr()
C_ut = sp.triu(C, k=1).tocoo()  # upper triangle to avoid duplicate edges
edges   = list(zip(C_ut.row.tolist(), C_ut.col.tolist()))
weights = C_ut.data.tolist()

g = ig.Graph(n=C.shape[0], edges=edges, directed=False)
g.es["weight"] = weights
Q = g.modularity(labels.tolist(), weights=weights)

# Collect, show, and save
extra_metrics = pd.DataFrame({
    "metric": [
        "Davies–Bouldin (lower better)",
        "Calinski–Harabasz (higher better)",
        "UMAP trustworthiness (higher better)",
        "Graph modularity (higher better)",
    ],
    "value": [db, ch, tw, Q],
})

display(extra_metrics.round(3))
extra_metrics.to_csv("/content/extra_clustering_metrics.csv", index=False)
print(f"[Part 10] DB={db:.3f} (lower), CH={ch:.1f} (higher), TW={tw:.3f} (higher), Q={Q:.3f} (higher)")



## **11) Denoising ×2 and impact on clustering/markers**

In [ ]:
# parameters (lightweight)
HVG_N   = 400     # number of HVGs to keep (panel size)
N_PCS   = 35      # PCs used in MAGIC + reconstruction
KNN_MAG = 12      # MAGIC KNN (smaller = lighter)
RES_LEI = float(str(best_key).split("_")[-1])  # reuse chosen Leiden resolution

# build a compact gene panel: HVGs ∪ ROC markers present in matrix
hv_mask = adata.var.get("highly_variable", pd.Series(False, index=adata.var_names))
hvg_genes_all = list(adata.var_names[hv_mask]) if hv_mask.any() else list(adata.var_names)
hvg_genes = hvg_genes_all[:HVG_N]

roc_all = list(adata.uns.get("roc_marker_candidates", []))
roc_in_matrix = [g for g in roc_all if g in set(map(str, adata.var_names))]

keep_genes = sorted(set(hvg_genes) | set(roc_in_matrix))
assert len(keep_genes) > 0, "No genes in the small panel; check HVG computation (Part 1) and ROC list (Part 5)."

# (A) MAGIC on the small panel (approximate solver)
ad_magic = adata[:, keep_genes].copy()
sc.external.pp.magic(
    ad_magic,
    n_pca=N_PCS,
    knn=KNN_MAG,
    t="auto",
    solver="approximate",
    random_state=0,
)

# graph/UMAP/Leiden on denoised small panel
sc.tl.pca(ad_magic, svd_solver="arpack", mask_var="highly_variable", random_state=0)
sc.pp.neighbors(ad_magic, n_neighbors=15, n_pcs=min(N_PCS, ad_magic.obsm["X_pca"].shape[1]), random_state=0)
sc.tl.umap(ad_magic, random_state=0)
sc.tl.leiden(ad_magic, resolution=RES_LEI, key_added="leiden_magic", random_state=0)

# compare to your Louvain (baseline part-3)
ari_magic = adjusted_rand_score(ad_magic.obs["leiden_magic"], adata.obs["louvain_igraph"])
nmi_magic = normalized_mutual_info_score(ad_magic.obs["leiden_magic"], adata.obs["louvain_igraph"])
sil_magic = silhouette_score(ad_magic.obsm["X_pca"], ad_magic.obs["leiden_magic"].cat.codes)

# (B) PCA low-rank reconstruction on the small panel
# reconstruct X (log1p) for keep_genes using original PCs
ad_pcad = adata[:, keep_genes].copy()
U_full  = adata.obsm["X_pca"][:, :N_PCS]                         # cells × PCs
V_full  = adata.varm["PCs"][:, :N_PCS]                           # genes(all) × PCs
idx     = adata.var_names.get_indexer(keep_genes)                # keep_genes - full matrix
V_panel = V_full[idx, :]                                         # genes(panel) × PCs
ad_pcad.X = U_full @ V_panel.T                                   # cells × genes(panel)

# graph/UMAP/Leiden after reconstruction
sc.tl.pca(ad_pcad, svd_solver="arpack", mask_var="highly_variable", random_state=0)
sc.pp.neighbors(ad_pcad, n_neighbors=15, n_pcs=min(N_PCS, ad_pcad.obsm["X_pca"].shape[1]), random_state=0)
sc.tl.umap(ad_pcad, random_state=0)
sc.tl.leiden(ad_pcad, resolution=RES_LEI, key_added="leiden_pca_denoise", random_state=0)

ari_pcad = adjusted_rand_score(ad_pcad.obs["leiden_pca_denoise"], adata.obs["louvain_igraph"])
nmi_pcad = normalized_mutual_info_score(ad_pcad.obs["leiden_pca_denoise"], adata.obs["louvain_igraph"])
sil_pcad = silhouette_score(ad_pcad.obsm["X_pca"], ad_pcad.obs["leiden_pca_denoise"].cat.codes)

# summary table (baseline part-4)
denoise_metrics = pd.DataFrame([
    {"label": "baseline",    "ARI": ari,         "NMI": nmi,         "Silhouette(PCA)": sil},
    {"label": "MAGIC_small", "ARI": ari_magic,   "NMI": nmi_magic,   "Silhouette(PCA)": sil_magic},
    {"label": "PCA_small",   "ARI": ari_pcad,    "NMI": nmi_pcad,    "Silhouette(PCA)": sil_pcad},
])
display(denoise_metrics)
denoise_metrics.to_csv("/content/denoise_metrics_compare.csv", index=False)

# marker stability (FAIR overlap restricted to small-panel genes)
roc_str = str(roc_cluster)
# baseline top-50 part-7 (computed on full space)
base_top50 = (markers_df.query("group == @roc_str")
                        .sort_values("scores", ascending=False)
                        .head(50)["names"].str.lower().tolist())

# fair baseline: only genes that exist in the small panel
keep_lower = set(map(str.lower, keep_genes))
base_top50_fair = [g for g in base_top50 if g in keep_lower]

def top50_for_variant(adx, leiden_key, score_col="ROC_score_tmp", top_n=50):
    # score only on the panel; pick the cluster with highest mean score
    sc.tl.score_genes(adx, gene_list=roc_in_matrix, score_name=score_col)
    target = adx.obs.groupby(leiden_key, observed=True)[score_col].mean().idxmax()
    # DE on the panel itself (use_raw=False since these are panel objects)
    sc.tl.rank_genes_groups(
        adx, groupby=leiden_key, groups=[target], reference="rest",
        method="wilcoxon", use_raw=False, n_genes=max(500, top_n), pts=True
    )
    df = (sc.get.rank_genes_groups_df(adx, group=str(target))
            .sort_values("scores", ascending=False)
            .head(top_n))
    return str(target), df["names"].str.lower().tolist()

roc_magic, magic_top50 = top50_for_variant(ad_magic, "leiden_magic")
roc_pcad,  pcad_top50  = top50_for_variant(ad_pcad,  "leiden_pca_denoise")

def jacc(a, b):
    A, B = set(a), set(b)
    return len(A & B) / max(1, len(A | B))

rows = []
if magic_top50:
    rows.append({
        "variant": "MAGIC_small",
        "overlap_top50": len(set(base_top50_fair) & set(magic_top50)),
        "jaccard_top50": jacc(base_top50_fair, magic_top50),
        "roc_cluster_variant": roc_magic
    })
if pcad_top50:
    rows.append({
        "variant": "PCA_small",
        "overlap_top50": len(set(base_top50_fair) & set(pcad_top50)),
        "jaccard_top50": jacc(base_top50_fair, pcad_top50),
        "roc_cluster_variant": roc_pcad
    })

overlap_df = pd.DataFrame(rows)
display(overlap_df)
overlap_df.to_csv(f"/content/denoise_marker_overlap_vs_baseline_{best_key}_ROC{roc_str}.csv", index=False)

# figures (saved to scanpy's figdir)
sc.pl.umap(ad_magic, color=["leiden_magic"],        wspace=0.4, save="_part11_magic_small.png", show=False)
sc.pl.umap(ad_pcad,  color=["leiden_pca_denoise"],  wspace=0.4, save="_part11_pca_small.png",   show=False)


## **12) Batch integration ×2 (Harmony, BBKNN) and impact on clustering/markers**



In [ ]:
import gc, logging, warnings

# quiet logs/warnings
logging.getLogger("harmonypy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="invalid value encountered in log2", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2  # compact Scanpy logging

# inputs carried from earlier parts
base_key  = best_key
batch_key = "batch"                  # column present in adata.obs
assert batch_key in adata.obs, "Missing 'batch' in adata.obs"
assert "X_pca" in adata.obsm, "Run PCA in Part 2 before Part 12."
roc_genes_all = list(adata.uns.get("roc_marker_candidates", []))
roc_genes = [g for g in roc_genes_all if g in set(map(str, adata.var_names))]
assert len(roc_genes) > 0, "roc_marker_candidates empty; re-run Part 5."

# make RAM-light object with only PCA + needed obs
use_pcs = min(30, adata.obsm["X_pca"].shape[1])   # 30 PCs keeps Harmony/BBKNN light
Xpca = adata.obsm["X_pca"][:, :use_pcs].astype(np.float32, copy=True)
ad_base = sc.AnnData(
    X=Xpca,  # keep PCA also in .X to avoid extra copies
    obs=adata.obs[[batch_key, base_key]].copy(),
)
ad_base.obsm["X_pca"] = Xpca
ad_base.obs_names = adata.obs_names.copy()
ad_base.obs["leiden_baseline"] = pd.Categorical(ad_base.obs[base_key]).astype(str)

# (A) Harmony on slim PCA (harmonypy)
# harmonypy signature: run_harmony(data_mat, meta_data, vars_use, ...)
meta = pd.DataFrame({batch_key: ad_base.obs[batch_key].astype(str).values},
                    index=ad_base.obs_names)
hobj = hm.run_harmony(
    ad_base.obsm["X_pca"],  # cells × PCs
    meta,
    [batch_key],
    max_iter_harmony=20,    # safe-with-30PCs
    theta=2.0,              # mild diversity penalty
    verbose=False,
)
ad_harm = ad_base.copy()
ad_harm.obsm["X_pca_harmony"] = hobj.Z_corr.T.astype(np.float32, copy=False)
sc.pp.neighbors(ad_harm, use_rep="X_pca_harmony", n_neighbors=15, random_state=0)
sc.tl.umap(ad_harm, random_state=0)
sc.tl.leiden(ad_harm, resolution=float(str(base_key).split("_")[-1]),
             key_added="leiden_harmony", random_state=0)

# (B) BBKNN on slim PCA
ad_bb = ad_base.copy()
bbknn.bbknn(ad_bb, batch_key=batch_key, neighbors_within_batch=7, n_pcs=use_pcs)
sc.tl.umap(ad_bb, random_state=0)
sc.tl.leiden(ad_bb, resolution=float(str(base_key).split("_")[-1]),
             key_added="leiden_bbknn", random_state=0)

# Metrics vs baseline (agreement + cluster silhouette + batch silhouette)
def _metrics_vs_baseline(ad_in, leiden_col, emb_key):
    L_base = pd.Categorical(ad_base.obs["leiden_baseline"])
    L_var  = pd.Categorical(ad_in.obs[leiden_col])
    X = ad_in.obsm[emb_key]
    ari = adjusted_rand_score(L_base.codes, L_var.codes)
    nmi = normalized_mutual_info_score(L_base.codes, L_var.codes)
    sil = silhouette_score(X, L_var.codes)
    bcodes = pd.Categorical(ad_in.obs[batch_key]).codes
    asw_batch = silhouette_score(X, bcodes)  # closer to 0 ⇒ better batch mixing
    return ari, nmi, sil, asw_batch

rows = []
m_base = _metrics_vs_baseline(ad_base, "leiden_baseline", "X_pca")
rows.append({"variant":"baseline","ARI_vs_baseline":m_base[0],"NMI_vs_baseline":m_base[1],
             "Silhouette_clusters":m_base[2],"Silhouette_batch":m_base[3]})
m_harm = _metrics_vs_baseline(ad_harm, "leiden_harmony", "X_pca_harmony")
rows.append({"variant":"harmony","ARI_vs_baseline":m_harm[0],"NMI_vs_baseline":m_harm[1],
             "Silhouette_clusters":m_harm[2],"Silhouette_batch":m_harm[3]})
m_bb = _metrics_vs_baseline(ad_bb, "leiden_bbknn", "X_pca")
rows.append({"variant":"bbknn","ARI_vs_baseline":m_bb[0],"NMI_vs_baseline":m_bb[1],
             "Silhouette_clusters":m_bb[2],"Silhouette_batch":m_bb[3]})

batch_metrics = pd.DataFrame(rows)
display(batch_metrics)
batch_metrics.to_csv("/content/batch_integration_metrics.csv", index=False)

# Marker stability (top-N) on a compact HVG∪ROC panel (RAM-safe)
# 1) bring variant labels back to the *full* object (aligned by index)
adata.obs["leiden_baseline"] = ad_base.obs["leiden_baseline"].reindex(adata.obs_names).astype(str)
adata.obs["leiden_bbknn"]    = ad_bb.obs["leiden_bbknn"].reindex(adata.obs_names).astype(str)
adata.obs["leiden_harmony"]  = ad_harm.obs["leiden_harmony"].reindex(adata.obs_names).astype(str)

# 2) build compact gene panel = HVGs ∪ ROC candidates
hvg_mask = adata.var.get("highly_variable", pd.Series(False, index=adata.var_names)).values
keep_genes = set(adata.var_names[hvg_mask].tolist()) | set(roc_genes)
keep_genes = [g for g in adata.var_names if g in keep_genes]
ad_small = adata[:, keep_genes].copy()
for col in ["leiden_baseline","leiden_bbknn","leiden_harmony"]:
    ad_small.obs[col] = adata.obs[col].reindex(ad_small.obs_names).astype(str)

def _labels_cat(series):
    cat = pd.Categorical(series)
    return cat.codes.astype(int), list(cat.categories.astype(str))

def _mean_by_label(values_1d, labels_codes, n_labels):
    sums = np.bincount(labels_codes, weights=values_1d, minlength=n_labels)
    cnts = np.bincount(labels_codes, minlength=n_labels).astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnts > 0, sums / cnts, 0.0)

def _topN_for(ad_small_in, leiden_col, gene_list, top_n=30):
    # score ROC markers; pick cluster with highest mean score
    sc.tl.score_genes(ad_small_in, gene_list=gene_list, score_name="ROC_tmp")
    codes, cats = _labels_cat(ad_small_in.obs[leiden_col])
    means = _mean_by_label(np.asarray(ad_small_in.obs["ROC_tmp"], float), codes, len(cats))
    target = cats[int(np.nanargmax(means))]
    # rank on the panel itself (use_raw=False)
    sc.tl.rank_genes_groups(ad_small_in, groupby=leiden_col, groups=[target], reference="rest",
                            method="wilcoxon", use_raw=False, n_genes=400, pts=True)
    rgg = ad_small_in.uns["rank_genes_groups"]["names"]
    if getattr(rgg, "dtype", None) and rgg.dtype.names:          # structured array
        genes = np.array(rgg[target])[:top_n]
    else:                                                         # (N_genes, N_groups) array
        groups = ad_small_in.uns["rank_genes_groups"]["groups"]
        gi = groups.index(target)
        genes = np.array(rgg)[:, gi][:top_n]
    return target, [str(g).lower() for g in genes]

top_n = 30
tgt_base, base_topN = _topN_for(ad_small, "leiden_baseline", roc_genes, top_n=top_n)
tgt_harm, harm_topN = _topN_for(ad_small, "leiden_harmony",  roc_genes, top_n=top_n)
tgt_bb,   bb_topN   = _topN_for(ad_small, "leiden_bbknn",    roc_genes, top_n=top_n)

def _jacc(a, b):
    A, B = set(a), set(b)
    return len(A & B) / max(1, len(A | B))

overlap_df = pd.DataFrame([
    {"variant":"Harmony","overlap_topN":len(set(base_topN)&set(harm_topN)),
     "jaccard_topN":_jacc(base_topN, harm_topN),
     "baseline_target":tgt_base, "variant_target":tgt_harm},
    {"variant":"BBKNN","overlap_topN":len(set(base_topN)&set(bb_topN)),
     "jaccard_topN":_jacc(base_topN, bb_topN),
     "baseline_target":tgt_base, "variant_target":tgt_bb},
])
display(overlap_df)
overlap_df.to_csv(f"/content/batch_integration_marker_overlap_{base_key}_top{top_n}.csv", index=False)

# Figures (saved to scanpy figdir)
sc.pl.umap(ad_harm, color=["leiden_harmony", batch_key], wspace=0.4, save="_harmony.png", show=False)
sc.pl.umap(ad_bb,   color=["leiden_bbknn",   batch_key], wspace=0.4, save="_bbknn.png",   show=False)

# Round & save tidy artifacts, persist labels, and collect figures
batch_metrics_rounded = (batch_metrics.copy()
                         .assign(**{
                             "ARI_vs_baseline":     batch_metrics["ARI_vs_baseline"].round(3),
                             "NMI_vs_baseline":     batch_metrics["NMI_vs_baseline"].round(3),
                             "Silhouette_clusters": batch_metrics["Silhouette_clusters"].round(3),
                             "Silhouette_batch":    batch_metrics["Silhouette_batch"].round(3),
                         }))
display(batch_metrics_rounded)
batch_metrics_rounded.to_csv("/content/batch_integration_metrics_rounded.csv", index=False)

overlap_rounded = overlap_df.copy()
overlap_rounded["jaccard_topN"] = overlap_rounded["jaccard_topN"].astype(float).round(3)
display(overlap_rounded)
overlap_rounded.to_csv("/content/batch_integration_marker_overlap_rounded.csv", index=False)

# Persist labels as categorical and save a single H5AD for grading
for col in ["leiden_baseline", "leiden_harmony", "leiden_bbknn"]:
    if col in adata.obs:
        adata.obs[col] = adata.obs[col].astype("category")
adata.write_h5ad("/content/frogtail_part12_final.h5ad", compression="gzip")

# Copy figures into one folder
figdir = Path(sc.settings.figdir or "figures")
outdir = Path("/content/frogtail_outputs/figures")
outdir.mkdir(parents=True, exist_ok=True)
import shutil, glob
for pat in ["umap_harmony.png", "umap_bbknn.png",
            "umap_Figure1_clustering.png", "umap_Figure2_ROCscore.png",
            "violin_Figure2_ROCscore_violin.png",
            "umap_part11_magic_small.png", "umap_part11_pca_small.png"]:
    for src in glob.glob(str(figdir / pat)):
        try:
            shutil.copy(src, outdir / Path(src).name)
        except Exception:
            pass

print("[Part 12] Saved:")
print(" - /content/batch_integration_metrics.csv and _rounded.csv")
print(" - /content/batch_integration_marker_overlap_*.csv")
print(" - /content/frogtail_part12_final.h5ad")
print(f" - Figures copied to: {outdir}")

# cleanup
del ad_base, ad_harm, ad_bb, ad_small, Xpca
gc.collect()


## **13) Compare ROC gene set to Supplementary Table 3**

In [ ]:
supp_path   = Path("/content/aav9996_tables3.xlsx")   # <- adjust if needed
target_sheet = "ROC markers"                          # <- preferred sheet name
top_n       = 100

assert supp_path.exists(), f"File not found: {supp_path}"

# Load Supplement sheet (first column) with a simple fallback
try:
    df_supp = pd.read_excel(supp_path, sheet_name=target_sheet)
    print(f"[Using sheet] {target_sheet}")
except Exception as e:
    xls = pd.ExcelFile(supp_path)
    print(f"[Harmony skipped] {e}")  # harmless message reuse; means target sheet missing
    print("[Suppl sheets]", xls.sheet_names)
    df_supp = pd.read_excel(supp_path, sheet_name=0)
    print("[Falling back to sheet 0]")

# Take the first column as the list of genes
supp_col = df_supp.columns[0]
supp_genes = df_supp[supp_col].astype(str)

# Quick peek from the sheet used
print("[First 10 rows, sheet 0]\n", pd.read_excel(supp_path, sheet_name=0).head(10))

# Build lowercase list and map to symbols present in your matrix (allow .l/.s)
vnames = pd.Index(adata.var_names)
vlow   = pd.Index([v.lower() for v in vnames])

def present_any(lower_name: str):
    """Return an exact var_name from adata for a generic name, tolerating .l/.s allele suffixes."""
    base = str(lower_name).strip().lower()
    for cand in (base, f"{base}.l", f"{base}.s"):
        if cand in vlow:
            return vnames[vlow.get_loc(cand)]
    return None

supp_low = (supp_genes.str.strip().str.lower().dropna().unique().tolist())
supp_in_matrix = [present_any(g) for g in supp_low]
supp_in_matrix = sorted({g for g in supp_in_matrix if g is not None})

# Baseline ROC markers (top-N) for your selected ROC cluster
roc_str = str(roc_cluster)
base_top = (
    markers_df.query("group == @roc_str")
              .sort_values("scores", ascending=False)
              .head(top_n)["names"].str.lower().tolist()
)

# Strict overlap / Jaccard / hypergeometric p-value
S = set([g.lower() for g in supp_in_matrix])  # supplement genes present in your matrix
B = set(base_top)                             # baseline top-N (lowercased)

overlap = len(S & B)
jaccard = overlap / max(1, len(S | B))

M = len(vnames)   # universe = genes in matrix
K = len(S)        # supp genes present in matrix
n = len(B)        # baseline top-N size
x = overlap       # observed overlap
pval = hypergeom.sf(x - 1, M, K, n)

compare_df = pd.DataFrame({
    "metric": ["supp_genes_in_matrix", "baseline_topN", "overlap", "jaccard", "hypergeom_p"],
    "value":  [len(S),                 len(B),          overlap,   jaccard,   pval]
})
display(compare_df)
compare_df.to_csv(f"/content/compare_supptable3_metrics_top{top_n}.csv", index=False)

# Canonicalized comparison (strip .l/.s suffixes)
def canon(x: str):
    x = str(x).lower().strip()
    return x[:-2] if x.endswith(".l") or x.endswith(".s") else x

S_canon = {canon(g) for g in S}
B_canon = {canon(g) for g in B}
overlap_canon = len(S_canon & B_canon)
jaccard_canon = overlap_canon / max(1, len(S_canon | B_canon))
print(f"[Canonicalized compare] overlap={overlap_canon}, jaccard={jaccard_canon:.3f}")

# Optional: show a few near-miss examples that only differ by allele suffix
near_miss = sorted((S ^ B) & ({g+".l" for g in B_canon} | {g+".s" for g in B_canon}))
if near_miss:
    print("[Near-miss after stripping .l/.s] e.g.:", near_miss[:3])
else:
    print("[No near-miss matches after canonicalization]")

# Save lists to tidy CSVs (one column each; safe even if lengths differ)
# Strict lists
pd.DataFrame({"gene": sorted(S)}).to_csv(
    f"/content/compare_supptable3_list_supp_in_matrix.csv", index=False
)
pd.DataFrame({"gene": sorted(B)}).to_csv(
    f"/content/compare_supptable3_list_baseline_top{top_n}.csv", index=False
)
pd.DataFrame({"gene": sorted(S & B)}).to_csv(
    f"/content/compare_supptable3_overlap_genes_top{top_n}.csv", index=False
)

# Canonicalized overlap list (useful for the report)
pd.DataFrame({"gene_canonical": sorted(S_canon & B_canon)}).to_csv(
    f"/content/compare_supptable3_overlap_genes_canonical_top{top_n}.csv", index=False
)

print(f"[Part 13] Wrote metrics + four CSVs (supp list, baseline top{top_n}, strict overlap, canonical overlap).")


## **14) Final figures**

In [ ]:
# Ensure fig dir exists
sc_figdir_main = Path(sc.settings.figdir if sc.settings.figdir else "figures")
sc_figdir_main.mkdir(parents=True, exist_ok=True)

#  Make sure required columns exist
# 1) louvain labels should exist from Part 3
if "louvain_igraph" not in adata.obs.columns:
    raise ValueError("Missing 'louvain_igraph' in adata.obs. Re-run Part 3.")

# 2) best_key from earlier (e.g., 'leiden_0.4')
if 'best_key' not in globals():
    # fall back to the first leiden_ present
    leiden_cols = [c for c in adata.obs.columns if str(c).startswith("leiden_")]
    if not leiden_cols:
        raise ValueError("No Leiden column found; re-run clustering.")
    best_key = leiden_cols[0]
print(f"[Part 14] Using Leiden key: {best_key}")

# 3) ROC_score from Part 5; if missing, compute from saved candidate list
if "ROC_score" not in adata.obs.columns:
    roc_candidates = list(adata.uns.get("roc_marker_candidates", []))
    if len(roc_candidates) == 0:
        print("[Part 14] No ROC_score and no saved candidates; skipping Figure 2.")
        want_fig2 = False
    else:
        sc.tl.score_genes(adata, gene_list=roc_candidates, score_name="ROC_score")
        want_fig2 = True
else:
    want_fig2 = True

# Figure 1: clustering summary (Leiden + Louvain)
# Saved as: figures/umap_Figure1_clustering.png
sc.pl.umap(
    adata,
    color=[best_key, "louvain_igraph"],
    wspace=0.4,
    save="_Figure1_clustering.png",
    show=False,
)
plt.close('all')
print(f"[Part 14] Saved Figure 1 -> {sc_figdir_main / 'umap_Figure1_clustering.png'}")

# Figure 2: ROC score map + violin by Leiden (if available)
if want_fig2:
    # UMAP with ROC score
    sc.pl.umap(
        adata,
        color=["ROC_score"],
        cmap="viridis",
        save="_Figure2_ROCscore.png",
        show=False,
    )
    plt.close('all')

    # Violin per Leiden cluster
    sc.pl.violin(
        adata,
        keys="ROC_score",
        groupby=best_key,
        rotation=90,
        stripplot=False,
        save="_Figure2_ROCscore_violin.png",
        show=False,
    )
    plt.close('all')

    print(f"[Part 14] Saved Figure 2 UMAP -> {sc_figdir_main / 'umap_Figure2_ROCscore.png'}")
    print(f"[Part 14] Saved Figure 2 Violin -> {sc_figdir_main / 'violin_Figure2_ROCscore_violin.png'}")
else:
    print("[Part 14] Skipped Figure 2 (no ROC_score).")

print("[Part 14] Done.")
